In [1]:
from ingest import load_corpus, embed_texts, chunk_documents, insert_documents, get_db_connection
from embedder import Embedder
from rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI
from config import FAQ_DATA_PATH, TIP_DATA_PATH

In [2]:
load_dotenv(override=True)
openai_client = OpenAI()

In [3]:
# ingest faq data and embed each topic
faq_documents = load_corpus(FAQ_DATA_PATH)
faq_embeddings = embed_texts([doc['topic'] for doc in faq_documents])

  0%|          | 0/4 [00:00<?, ?it/s]

In [4]:
# ingest tip data and embed each topic
tip_documents = load_corpus(TIP_DATA_PATH)
tip_chunks = chunk_documents(tip_documents, content_key="content", chunk_size=500, overlap=100)
tip_embeddings = embed_texts([f"{chunk['topic']}\n{chunk['content']}" for chunk in tip_chunks])

  0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
insert_documents(faq_documents, faq_embeddings)
insert_documents(tip_chunks, tip_embeddings)

In [10]:
conn = get_db_connection()
embedder = Embedder()
vector_assistant = RAGBase(
    embedder=embedder,
    conn=conn,
    llm_client=openai_client,
)

In [ ]:
query = "Comment pouvez-vous financer vos études supérieures ?"

In [14]:
vector_assistant.rag(query, num_results=5)

'Je ne sais pas.'

In [ ]:
# conn.close()